[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [NumPy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)

# Broadcasting &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import math
from datetime import datetime, timedelta

import numpy as np

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
START = datetime(2025, 1, 1)
HOURS = 365 * 24


def year_of_readings(station):
    """A reading for every hour of 2025 at one station, as the sqlite3 guide makes them.
    Svalbard was silent on 2 March, so those 24 hours are None."""
    i = list(STATIONS).index(station)
    row = []
    for n in range(HOURS):
        hour = START + timedelta(hours=n)
        if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
            row.append(None)
        else:
            season = -math.cos(2 * math.pi * (n - 400) / HOURS)
            day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
            wobble = ((n * 37 + i * 101) % 17 - 8) / 10
            row.append(round(STATIONS[station] + 9 * season + 3 * day + wobble, 1) + 0.0)
    return row


readings_list = [year_of_readings(station) for station in STATIONS]
readings = np.array(readings_list, dtype=float)       # None becomes nan


def hour_of_year(month, day, hour=0):
    """The position along a station's row of an hour of 2025. hour_of_year(7, 1, 12) is noon on 1 July."""
    return (datetime(2025, month, day, hour) - START) // timedelta(hours=1)


print("numpy", np.__version__, "|", len(readings_list), "stations,", len(readings_list[0]), "hours each |", readings.shape)

numpy 2.1.3 | 4 stations, 8760 hours each | (4, 8760)


**1.** Subtract Oslo's row, row 1, from every station, and print each station's mean difference from Oslo with `np.nanmean`, rounded to two places.

In [2]:
from_oslo = readings - readings[1]

print(np.nanmean(from_oslo, axis=1).round(2))

[  1.5   0.  -11.   -3. ]


Oslo's row has shape (8760,), which lines up with the hours. `np.nanmean` leaves out Svalbard's missing hours.

**2.** Use `np.broadcast_shapes` to print the shape that (365, 24) with (24,) gives, and the shape that (365, 24) with (365, 1) gives.

In [3]:
print(np.broadcast_shapes((365, 24), (24,)))
print(np.broadcast_shapes((365, 24), (365, 1)))

(365, 24)
(365, 24)


The first stretches a row of 24 down the 365 days, and the second a column of 365 across the 24 hours.

**3.** Compute each station's anomaly from its own mean with `keepdims=True`, and print the anomalies at noon on 1 January, rounded to two places.

In [4]:
station_means = np.nanmean(readings, axis=1, keepdims=True)
anomaly = readings - station_means

print(anomaly[:, hour_of_year(1, 1, 12)].round(2))

[-7.1  -7.2  -7.32 -5.7 ]


`keepdims=True` gives the means shape (4, 1), so each one meets its own station's row.

**4.** Count the hours below -10, -5 and 0 degrees at every station in one comparison, using a column of thresholds, and print the table.

In [5]:
limits = np.array([-10, -5, 0])

print((readings < limits[:, np.newaxis, np.newaxis]).sum(axis=2))

[[   0    0 2358    0]
 [   0  134 4179 1041]
 [1220 1848 5871 3203]]


The last row is each station's frost hours, the counts the **Boolean Masks** notebook found one station at a time.

**5.** Reshape Bergen's row to (365, 24), subtract the mean of each hour of the day across the year, and print the result's shape and its largest value, rounded to two places.

In [6]:
bergen_days = readings[0].reshape(365, 24)
hourly = bergen_days.mean(axis=0)
remainder = bergen_days - hourly

print(hourly.shape, remainder.shape)
print(remainder.max().round(2))

(24,) (365, 24)
9.82


`hourly` has shape (24,), which lines up with the hours of the day, so it needs no new axis.

---

&#8592; **Back to:** [Broadcasting](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/08-broadcasting.ipynb)  &nbsp;&middot;&nbsp;  [NumPy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)
